In [0]:
%pip install seaborn
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


df = spark.table('flight_delay_predictor_project.default.first_prediction_table').toPandas()

In [0]:
from sklearn.model_selection import train_test_split
df["weather_delay"] = df["weather_delay"].fillna(0)

In [0]:
df = spark.createDataFrame(df)

In [0]:
from pyspark.sql import Window, functions as F

hr_delay = (df.withColumn("delay_in_hr", 
F.when(F.col("weather_delay") < 1, "no delay")
.when(F.col("weather_delay") < 60, "less than 1") 
.when(F.col("weather_delay") < 120, "1+")
.when(F.col("weather_delay") < 180, "2+")
.when(F.col("weather_delay") < 240, "3+")
.when(F.col("weather_delay") < 300, "4+")
.when(F.col("weather_delay") < 360, "5+")
.when(F.col("weather_delay") < 420, "6+")
.when(F.col("weather_delay") < 480, "7+")
.when(F.col("weather_delay") < 540, "8+")
.when(F.col("weather_delay") < 600, "9+")
.otherwise("10+")) 
.select("temperature_2m", "relative_humidity_2m", "precipitation", "snowfall", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "cloud_cover", "pressure_msl", "weather_code", "delay_in_hr"))

In [0]:
hr_delay = hr_delay.toPandas()

In [0]:
flights_1 = hr_delay[hr_delay['delay_in_hr'] != 'no delay']
flights_2 = hr_delay[hr_delay['delay_in_hr'] == 'no delay']
print(flights_1.shape)
print(flights_2.shape)

Testing different upsampling amounts (might be point less need to ask dr song)

In [0]:
from sklearn.utils import resample

flights_1_upsampled = resample(flights_1,random_state=42,n_samples=101915,replace=True)
flights_upsampled = pd.concat([flights_1_upsampled,flights_2])

Testing different test_size

In [0]:
from sklearn.tree import DecisionTreeClassifier

X = flights_upsampled.drop('delay_in_hr',axis=1)
y = flights_upsampled['delay_in_hr']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,random_state=42)
dtree = DecisionTreeClassifier()

test_size=0.20 gave the best results

In [0]:
dtree.fit(X_train,y_train)

In [0]:
from sklearn.metrics import accuracy_score

predictions = dtree.predict(X_test)
#print(classification_report(y_test,predictions))
#print(confusion_matrix(y_test,predictions))
accuracy_score(y_test, predictions)

Feature Importance

In [0]:
from sklearn.ensemble import RandomForestClassifier

# This line instantiates the model. 
rf = RandomForestClassifier() 

# Fit the model on your training data.
rf.fit(X_train, y_train) 
y_predict = rf.predict(X_test)

# And score it on your testing data.
accuracy_score(y_test, y_predict)

In [0]:
feature_importances = pd.DataFrame(rf.feature_importances_,
                                   index = X_train.columns,
                                   columns=['importance']).sort_values('importance', ascending=False)
feature_importances

Testing Data on the top 7 features 

In [0]:
hr_delay = (df.withColumn("delay_in_hr", 
F.when(F.col("weather_delay") < 1, "no delay")
.when(F.col("weather_delay") < 60, "less than 1") 
.when(F.col("weather_delay") < 120, "1+")
.when(F.col("weather_delay") < 180, "2+")
.when(F.col("weather_delay") < 240, "3+")
.when(F.col("weather_delay") < 300, "4+")
.when(F.col("weather_delay") < 360, "5+")
.when(F.col("weather_delay") < 420, "6+")
.when(F.col("weather_delay") < 480, "7+")
.when(F.col("weather_delay") < 540, "8+")
.when(F.col("weather_delay") < 600, "9+")
.otherwise("10+")) 
.select("temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "pressure_msl", "weather_code", "delay_in_hr"))

In [0]:
hr_delay = hr_delay.toPandas()

flights_1 = hr_delay[hr_delay['delay_in_hr'] != 'no delay']
flights_2 = hr_delay[hr_delay['delay_in_hr'] == 'no delay']

flights_1_upsampled = resample(flights_1,random_state=42,n_samples=101915,replace=True)
flights_upsampled = pd.concat([flights_1_upsampled,flights_2])

X = flights_upsampled.drop('delay_in_hr',axis=1)
y = flights_upsampled['delay_in_hr']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,random_state=42)
dtree = DecisionTreeClassifier()

dtree.fit(X_train,y_train)


In [0]:
predictions = dtree.predict(X_test)
#print(classification_report(y_test,predictions))
#print(confusion_matrix(y_test,predictions))
accuracy_score(y_test, predictions)

Choosing the top 7 features did not make much of a difference as it only dropped by 0.0002 in accuracy, we will continue to use all 10 features when adding more hourly weather data to our testing table 

Down Sampling

In [0]:
flights_1 = hr_delay[hr_delay['delay_in_hr'] != 'no delay']
flights_2 = hr_delay[hr_delay['delay_in_hr'] == 'no delay']
print(flights_1.shape)
print(flights_2.shape)

In [0]:
from sklearn.utils import resample

flights_2_downsampled = resample(flights_2,random_state=42,n_samples=845,replace=True)
flights_downsampled = pd.concat([flights_2_downsampled,flights_1])

In [0]:
from sklearn.tree import DecisionTreeClassifier

X = flights_downsampled.drop('delay_in_hr',axis=1)
y = flights_downsampled['delay_in_hr']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30,random_state=42)
dtree = DecisionTreeClassifier()

In [0]:
dtree.fit(X_train,y_train)


In [0]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

predictions = dtree.predict(X_test)
print(classification_report(y_test,predictions))
print(confusion_matrix(y_test,predictions))
accuracy_score(y_test, predictions)

accuracy score was a lot worse with down sampling, which makes sense

Adding more columns to the data

In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.more_hourly_data_table AS
WITH flights_parsed AS (
  SELECT
    *,
    COALESCE(
      try_to_timestamp(CAST(FL_DATE AS STRING), 'M/d/yyyy h:mm:ss a'),
      try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd HH:mm:ss'),
      try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd')
    ) AS fl_date_ts,
    CAST(CRS_DEP_TIME AS INT) AS crs_dep_int
  FROM flight_delay_predictor_project.default.jfk_flight_data
),
flights_hourly AS (
  SELECT
    *,
    timestampadd(HOUR, crs_dep_int DIV 100, fl_date_ts) AS dep_hour,
    timestampadd(MINUTE, crs_dep_int % 100,
                 timestampadd(HOUR, crs_dep_int DIV 100, fl_date_ts)) AS crs_dep_ts
  FROM flights_parsed
)
SELECT
  f.*,
  -- Weather at departure hour
  w0.* EXCEPT (time),
  -- Weather 2 hour after
  w2.pressure_msl          AS pressure_msl_plus2h,
  w2.relative_humidity_2m  AS relative_humidity_2m_plus2h,
  w2.wind_direction_10m    AS wind_direction_10m_plus2h,
  w2.temperature_2m        AS temperature_2m_plus2h,
  w2.precipitation         AS precipitation_plus2h,
  w2.snowfall              AS snowfall_plus2h,
  w2.wind_speed_10m        AS wind_speed_10m_plus2h,
  w2.wind_gusts_10m        AS wind_gusts_10m_plus2h,
  w2.cloud_cover           AS cloud_cover_plus2h,
  w2.weather_code          AS weather_code_plus2h,
  -- Weather 5 hours after
  w5.temperature_2m        AS temperature_2m_plus5h,
  w5.precipitation         AS precipitation_plus5h,
  w5.snowfall              AS snowfall_plus5h,
  w5.wind_speed_10m        AS wind_speed_10m_plus5h,
  w5.wind_gusts_10m        AS wind_gusts_10m_plus5h,
  w5.cloud_cover           AS cloud_cover_plus5h,
  w5.weather_code          AS weather_code_plus5h,
  w5.wind_direction_10m    AS wind_direction_10m_plus5h,
  w5.pressure_msl          AS pressure_msl_plus5h,
  w5.relative_humidity_2m  AS relative_humidity_2m_plus5h,
    -- Weather 7 hours after
  w7.temperature_2m        AS temperature_2m_plus7h,
  w7.precipitation         AS precipitation_plus7h,
  w7.snowfall              AS snowfall_plus7h,
  w7.wind_speed_10m        AS wind_speed_10m_plus7h,
  w7.wind_gusts_10m        AS wind_gusts_10m_plus7h,
  w7.cloud_cover           AS cloud_cover_plus7h,
  w7.weather_code          AS weather_code_plus7h,
  w7.wind_direction_10m    AS wind_direction_10m_plus7h,
  w7.pressure_msl          AS pressure_msl_plus7h,
  w7.relative_humidity_2m  AS relative_humidity_2m_plus7h
FROM flights_hourly f
LEFT JOIN flight_delay_predictor_project.default.jfk_weather_history w0
  ON f.dep_hour = w0.time
LEFT JOIN flight_delay_predictor_project.default.jfk_weather_history w2
  ON timestampadd(HOUR, 2, f.dep_hour) = w2.time
LEFT JOIN flight_delay_predictor_project.default.jfk_weather_history w5
  ON timestampadd(HOUR, 5, f.dep_hour) = w5.time
LEFT JOIN flight_delay_predictor_project.default.jfk_weather_history w7
  ON timestampadd(HOUR, 7, f.dep_hour) = w7.time;

In [0]:
%sql
SELECT *
FROM flight_delay_predictor_project.default.more_hourly_data_table

In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.more_hourly_data_table_params 
SELECT weather_delay, temperature_2m, relative_humidity_2m, precipitation, snowfall, wind_speed_10m, wind_direction_10m, wind_gusts_10m, cloud_cover, pressure_msl, weather_code, temperature_2m_plus2h, precipitation_plus2h, snowfall_plus2h, wind_speed_10m_plus2h, wind_gusts_10m_plus2h, cloud_cover_plus2h, weather_code_plus2h, relative_humidity_2m_plus2h, wind_direction_10m_plus2h, pressure_msl_plus2h, temperature_2m_plus5h, precipitation_plus5h, snowfall_plus5h, wind_speed_10m_plus5h, wind_gusts_10m_plus5h, cloud_cover_plus5h, weather_code_plus5h, relative_humidity_2m_plus5h, wind_direction_10m_plus5h, pressure_msl_plus5h, temperature_2m_plus7h, precipitation_plus7h, snowfall_plus7h, wind_speed_10m_plus7h, wind_gusts_10m_plus7h, cloud_cover_plus7h, weather_code_plus7h, relative_humidity_2m_plus7h, wind_direction_10m_plus7h, pressure_msl_plus7h
FROM flight_delay_predictor_project.default.more_hourly_data_table
WHERE fl_date_ts < '2026-06-29';

In [0]:
%pip install seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from sklearn.utils import resample

In [0]:
df = spark.table('flight_delay_predictor_project.default.more_hourly_data_table_params').toPandas()

In [0]:
from sklearn.model_selection import train_test_split
df["weather_delay"] = df["weather_delay"].fillna(0)

In [0]:
from pyspark.sql import Window, functions as F

if isinstance(df, pd.DataFrame):
    df = spark.createDataFrame(df)

hr_delay = (df.withColumn("delay_in_hr", 
F.when(F.col("weather_delay") < 1, "no delay")
.when(F.col("weather_delay") < 60, "less than 1") 
.when(F.col("weather_delay") < 120, "1+")
.when(F.col("weather_delay") < 180, "2+")
.when(F.col("weather_delay") < 240, "3+")
.when(F.col("weather_delay") < 300, "4+")
.when(F.col("weather_delay") < 360, "5+")
.when(F.col("weather_delay") < 420, "6+")
.when(F.col("weather_delay") < 480, "7+")
.when(F.col("weather_delay") < 540, "8+")
.when(F.col("weather_delay") < 600, "9+")
.otherwise("10+")) 
.select("temperature_2m", "relative_humidity_2m", "precipitation", "snowfall", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "cloud_cover", "pressure_msl", "weather_code", "delay_in_hr",
"temperature_2m_plus2h", "precipitation_plus2h", "snowfall_plus2h", "wind_speed_10m_plus2h", "wind_gusts_10m_plus2h", "cloud_cover_plus2h", "weather_code_plus2h", "relative_humidity_2m_plus2h", "wind_direction_10m_plus2h", "pressure_msl_plus2h",
"temperature_2m_plus5h", "precipitation_plus5h", "snowfall_plus5h", "wind_speed_10m_plus5h", "wind_gusts_10m_plus5h", "cloud_cover_plus5h", "weather_code_plus5h", "relative_humidity_2m_plus5h", "wind_direction_10m_plus5h", "pressure_msl_plus5h",
"temperature_2m_plus7h", "precipitation_plus7h", "snowfall_plus7h", "wind_speed_10m_plus7h", "wind_gusts_10m_plus7h", "cloud_cover_plus7h", "weather_code_plus7h", "relative_humidity_2m_plus7h", "wind_direction_10m_plus7h", "pressure_msl_plus7h"))

In [0]:
from sklearn.tree import DecisionTreeClassifier

if not isinstance(hr_delay, pd.DataFrame):
    hr_delay = hr_delay.toPandas()

flights_1 = hr_delay[hr_delay['delay_in_hr'] != 'no delay']
flights_2 = hr_delay[hr_delay['delay_in_hr'] == 'no delay']

flights_1_upsampled = resample(flights_1,random_state=42,n_samples=101915,replace=True)
flights_upsampled = pd.concat([flights_1_upsampled,flights_2])

X = flights_upsampled.drop('delay_in_hr',axis=1)
y = flights_upsampled['delay_in_hr']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,random_state=42)
dtree = DecisionTreeClassifier()

dtree.fit(X_train,y_train)

In [0]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

predictions = dtree.predict(X_test)
print(classification_report(y_test,predictions))
print(confusion_matrix(y_test,predictions))
accuracy_score(y_test, predictions)

Trying to predict a how long a flight will be delayed for 10 hours is still too much so our next and final prediction is to test on the CRS take off time, 1+ hour weather data, 2+ hour weather data and predict weather the flight will have no weather delay, less than 1 hour, 2-4 hours, or 5+ hour delay